<a href="https://colab.research.google.com/github/tamzinzanalcock/Latin-Stylometry/blob/main/Latin_Stylometry_2_word_for_word.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cltk

In [ ]:
from cltk.data.fetch import FetchCorpus

corpus_downloader = FetchCorpus(language="lat")
corpus_downloader.import_corpus("lat_models_cltk")

INFO:CLTK:Cloning 'lat_models_cltk' from 'https://github.com/cltk/lat_models_cltk.git'


In [ ]:
from cltk.lemmatize.lat import LatinBackoffLemmatizer
lemmatizer = LatinBackoffLemmatizer()

In [ ]:
!pip install stanza
!python -c "import stanza; stanza.download('la')"

2026-08-08 14:58:49 INFO: Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
2026-08-08 14:58:49 INFO: Downloading default packages for language: la (Latin) ...

models/default.zip: downloading bytes:   0% 193k/224M [00:00<18:46, 199kB/s]
models/default.zip: downloading bytes:  10% 21.6M/224M [00:01<00:06, 30.7MB/s, 1.52MB/s  ]
models/default.zip: downloading bytes:  36% 80.6M/224M [00:03<00:03, 38.3MB/s, 6.39MB/s  ]
models/default.zip: downloading bytes:  39% 86.5M/224M [00:03<00:03, 43.2MB/s, 6.83MB/s  ]
models/default.zip: downloading bytes:  98% 221M/224M [00:04<00:00, 123MB/s, 17.9MB/s  ]
models/default.zip: reconstructing file:  70% 158M/224M [00:04<00:01, 58.0MB/s, 2.20MB/s  ] 
models/default.zip: downloading bytes: 100% 224M/224M [00:04<00:00, 45.7MB/s, 18.3MB/s  ]
models/default.zip: reconstructing file: 100% 224M/224M [00:04<00:00, 45.8MB/s, 19.7MB/s  ]
2026-08-08 14:58:55 INFO: Downloaded file to /root/.cache/stanza/1.14.0/resources/la/default.zip
2026-08

In [ ]:
from collections import Counter
import string
import re
import unicodedata




# Words where -que/-ve/-ne is NOT an enclitic (part of the word itself)
ENCLITIC_EXCEPTIONS = {
    'que', 'atque', 'neque', 'quoque', 'itaque', 'absque', 'undique',
    'utique', 'denique', 'quandoque', 'ubique', 'plerique', 'plerumque',
    've', 'sive', 'neve',
    'ne', 'bene', 'plane', 'sane', 'paene', 'pone', 'sine', 'fere',
}

# Function words grouped by lemma, with common inflected forms.
# Not exhaustive (Latin morphology is large), but covers the
# high-frequency forms that actually show up in classical prose.
LATIN_FUNCTION_WORD_LEMMAS = {
    # --- conjunctions / particles (indeclinable, no inflection needed) ---
    'et':     {'et'},
    'atque':  {'atque', 'ac'},
    'sed':    {'sed'},
    'aut':    {'aut'},
    'vel':    {'vel'},
    'nam':    {'nam'},
    'enim':   {'enim'},
    'igitur': {'igitur'},
    'ergo':   {'ergo'},
    'tamen':  {'tamen'},
    'autem':  {'autem'},
    'que':    {'que'},
    've':     {'ve'},
    'si':     {'si'},
    'ut':     {'ut'},
    'ne':     {'ne'},
    'non':    {'non'},
    'quia':   {'quia'},
    'quam':   {'quam'},

    # --- prepositions (indeclinable) ---
    'in':   {'in'},
    'ad':   {'ad'},
    'cum':  {'cum'},
    'ex':   {'ex', 'e'},
    'de':   {'de'},
    'ab':   {'ab', 'a'},

    # --- sum, esse (to be) — very high frequency, heavily inflected ---
    'sum': {
        'sum', 'es', 'est', 'sumus', 'estis', 'sunt',        # present
        'eram', 'eras', 'erat', 'eramus', 'eratis', 'erant', # imperfect
        'ero', 'eris', 'erit', 'erimus', 'eritis', 'erunt',  # future
        'fui', 'fuisti', 'fuit', 'fuimus', 'fuistis', 'fuerunt', 'fuere', # perfect
        'sim', 'sis', 'sit', 'simus', 'sitis', 'sint',       # subjunctive
        'esse', 'fuisse', 'futurus',                          # infinitives/participle
    },

    # --- is, ea, id (he/she/it/that) — demonstrative/anaphoric pronoun ---
    'is': {
        'is', 'ea', 'id', 'eius', 'ei', 'eum', 'eam',
        'eo', 'ea', 'ii', 'eae', 'eorum', 'earum',
        'iis', 'eis', 'eos', 'eas',
    },

    # --- hic, haec, hoc (this) ---
    'hic': {
        'hic', 'haec', 'hoc', 'huius', 'huic', 'hunc', 'hanc',
        'hoc', 'hi', 'hae', 'horum', 'harum', 'his', 'hos', 'has',
    },

    # --- ille, illa, illud (that) ---
    'ille': {
        'ille', 'illa', 'illud', 'illius', 'illi', 'illum', 'illam',
        'illo', 'illi', 'illae', 'illorum', 'illarum',
        'illis', 'illos', 'illas',
    },

    # --- ipse, ipsa, ipsum (himself/itself) ---
    'ipse': {
        'ipse', 'ipsa', 'ipsum', 'ipsius', 'ipsi', 'ipsum', 'ipsam',
        'ipso', 'ipsi', 'ipsae', 'ipsorum', 'ipsarum',
        'ipsis', 'ipsos', 'ipsas',
    },

    # --- qui, quae, quod (relative/interrogative pronoun) ---
    'qui': {
        'qui', 'quae', 'quod', 'cuius', 'cui', 'quem', 'quam',
        'quo', 'qua', 'quorum', 'quarum',
        'quibus', 'quos', 'quas',
    },

    # --- sui, suus (reflexive) ---
    'sui': {'sui', 'sibi', 'se'},
    'suus': {
        'suus', 'sua', 'suum', 'sui', 'suae', 'suo',
        'suos', 'suas', 'suorum', 'suarum', 'suis',
    },
}

# Flatten into: form -> lemma  (fast lookup when scanning clean_words)
FORM_TO_LEMMA = {
    form: lemma
    for lemma, forms in LATIN_FUNCTION_WORD_LEMMAS.items()
    for form in forms
}




def normalize_macrons(word):
    """Strip macrons and other diacritics, e.g. ū -> u, ō -> o."""
    normalized = unicodedata.normalize('NFD', word)
    stripped = ''.join(ch for ch in normalized if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', stripped)


def split_enclitics(word):
    """Split -que, -ve, -ne enclitics off a word, returning (stem, enclitic or None)."""
    if word in ENCLITIC_EXCEPTIONS:
        return word, None

    for enclitic in ('que', 've', 'ne'):
        if word.endswith(enclitic) and len(word) > len(enclitic) + 2:
            stem = word[:-len(enclitic)]
            return stem, enclitic

    return word, None


def function_word_profile(clean_words, form_to_lemma=FORM_TO_LEMMA):
    """Relative frequency of each function-word LEMMA (proportion of total words)."""
    total = len(clean_words)
    lemma_counts = Counter()
    for w in clean_words:
        lemma = form_to_lemma.get(w)
        if lemma:
            lemma_counts[lemma] += 1
    profile = {lemma: count / total for lemma, count in lemma_counts.items()}
    return dict(sorted(profile.items(), key=lambda x: -x[1]))


def parse_feats(feats_str):
    """Turn 'Case=Gen|Number=Sing|Gender=Masc' into a dict."""
    if not feats_str:
        return {}
    return dict(pair.split('=') for pair in feats_str.split('|'))


def morphological_profile(text):
    doc = nlp(text)

    case_counts = Counter()
    tense_counts = Counter()
    mood_counts = Counter()
    voice_counts = Counter()
    pos_counts = Counter()
    tagged_words = []

    for sent in doc.sentences:
        for word in sent.words:
            feats = parse_feats(word.feats)
            pos_counts[word.upos] += 1
            if 'Case' in feats:
                case_counts[feats['Case']] += 1
            if 'Tense' in feats:
                tense_counts[feats['Tense']] += 1
            if 'Mood' in feats:
                mood_counts[feats['Mood']] += 1
            if 'Voice' in feats:
                voice_counts[feats['Voice']] += 1
            tagged_words.append((word.text, word.lemma, word.upos, feats))

    total = len(tagged_words)
    return {
        "tagged_words": tagged_words,
        "pos_distribution": {k: v / total for k, v in pos_counts.items()} if total else {},
        "case_distribution": {k: v / sum(case_counts.values()) for k, v in case_counts.items()} if case_counts else {},
        "tense_distribution": {k: v / sum(tense_counts.values()) for k, v in tense_counts.items()} if tense_counts else {},
        "mood_distribution": {k: v / sum(mood_counts.values()) for k, v in mood_counts.items()} if mood_counts else {},
        "voice_distribution": {k: v / sum(voice_counts.values()) for k, v in voice_counts.items()} if voice_counts else {},
    }

!pip install stanza
import stanza
stanza.download('la')
nlp = stanza.Pipeline('la', processors='tokenize,pos,lemma,depparse')

def print_word_by_word(stats):
    """Print every word with its lemma, POS, and morphological features."""
    if "tagged_words" not in stats:
        print("No morphological data — make sure morphology_flag=True was used.")
        return

    for word, lemma, pos, feats in stats["tagged_words"]:
        if feats:
            feats_str = ", ".join(f"{k}={v}" for k, v in feats.items())
        else:
            feats_str = "—"
        print(f"{word:<15} lemma={lemma:<12} pos={pos:<6} {feats_str}")

def analyze_text(text, label="Text", split_enclitics_flag=True, normalize_flag=True, lemmatize_flag=True, morphology_flag=True):
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]

    words = text.split()
    pre_clean_words = []
    enclitics_found = []

    for word in words:
        word = word.strip(string.punctuation).lower()
        if not word:
            continue
        if normalize_flag:
            word = normalize_macrons(word)
        if split_enclitics_flag:
            stem, enclitic = split_enclitics(word)
            pre_clean_words.append(stem)
            if enclitic:
                enclitics_found.append(enclitic)
        else:
            pre_clean_words.append(word)

    # Lemmatize in one batch call (much faster than per-word)
    if lemmatize_flag:
        lemma_pairs = lemmatizer.lemmatize(pre_clean_words)
        clean_words = [lemma for (form, lemma) in lemma_pairs]
    else:
        clean_words = pre_clean_words

    frequencies = Counter(clean_words)
    word_lengths = [len(w) for w in clean_words]

    stats = {
        "label": label,
        "total_words": len(clean_words),
        "unique_words": len(frequencies),
        "vocab_diversity": len(frequencies) / len(clean_words) if clean_words else 0,
        "avg_word_length": sum(word_lengths) / len(word_lengths) if word_lengths else 0,
        "num_sentences": len(sentences),
        "avg_sentence_length": len(clean_words) / len(sentences) if sentences else 0,
        "most_common": frequencies.most_common(5),
        "frequencies": frequencies,
        "enclitic_count": len(enclitics_found),
        "enclitic_breakdown": Counter(enclitics_found),
       "function_word_profile": function_word_profile(clean_words),
    }

    if morphology_flag:
        morph = morphological_profile(text)
        stats["pos_distribution"] = morph["pos_distribution"]
        stats["case_distribution"] = morph["case_distribution"]
        stats["tense_distribution"] = morph["tense_distribution"]
        stats["mood_distribution"] = morph["mood_distribution"]
        stats["voice_distribution"] = morph["voice_distribution"]
        stats["tagged_words"] = morph["tagged_words"]

    return stats


def print_stats(stats):
    print(f"--- {stats['label']} ---")
    print("Total words:", stats["total_words"])
    print("Different words:", stats["unique_words"])
    print("Vocabulary diversity:", round(stats["vocab_diversity"], 4))
    print("Average word length:", round(stats["avg_word_length"], 2))
    print("Number of sentences:", stats["num_sentences"])
    print("Average sentence length (words):", round(stats["avg_sentence_length"], 2))
    print("Most common words:", stats["most_common"])
    print("Enclitics found:", stats["enclitic_count"], dict(stats["enclitic_breakdown"]))
    print("Function-word profile (top 5):",
          dict(list(stats["function_word_profile"].items())[:5]))

    if "case_distribution" in stats:
        print("POS distribution:", {k: round(v, 3) for k, v in stats["pos_distribution"].items()})
        print("Case distribution:", {k: round(v, 3) for k, v in stats["case_distribution"].items()})
        print("Tense distribution:", {k: round(v, 3) for k, v in stats["tense_distribution"].items()})
        print("Mood distribution:", {k: round(v, 3) for k, v in stats["mood_distribution"].items()})
        print("Voice distribution:", {k: round(v, 3) for k, v in stats["voice_distribution"].items()})

    print()




text1 = """

"""

stats1 = analyze_text(text1, label="Augustus excerpt")
print_stats(stats1)
print_word_by_word(stats1)


INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Downloading default packages for language: la (Latin) ...
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/la/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.14.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Loading these models for language: la (Latin):
| Processor | Package       |
-----------------------------
| tokenize  | ittb          |
| mwt       | ittb          |
| pos       | ittb_nocharlm |
| lemma     | ittb_nocharlm |
| depparse  | ittb_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


--- Augustus excerpt ---
Total words: 56
Different words: 43
Vocabulary diversity: 0.7679
Average word length: 5.73
Number of sentences: 3
Average sentence length (words): 18.67
Most common words: [('et', 7), ('non', 3), ('praesens', 2), ('ego', 2), ('defero', 2)]
Enclitics found: 0 {}
Function-word profile (top 5): {'et': 0.125, 'non': 0.05357142857142857, 'ab': 0.03571428571428571, 'sum': 0.017857142857142856, 'in': 0.017857142857142856}
POS distribution: {'NOUN': 0.238, 'CCONJ': 0.111, 'VERB': 0.19, 'PRON': 0.032, 'ADP': 0.063, 'PUNCT': 0.111, 'PROPN': 0.032, 'PART': 0.063, 'AUX': 0.016, 'ADJ': 0.048, 'SCONJ': 0.032, 'ADV': 0.016, 'DET': 0.048}
Case distribution: {'Acc': 0.312, 'Dat': 0.094, 'Abl': 0.438, 'Nom': 0.094, 'Gen': 0.062}
Tense distribution: {'Fut': 0.167, 'Past': 0.333, 'Pres': 0.5}
Mood distribution: {'Ind': 1.0}
Voice distribution: {'Act': 0.667, 'Pass': 0.333}

Dictaturam      lemma=dictatura    pos=NOUN   Case=Acc, Gender=Fem, InflClass=IndEurA, Number=Sing
et       